In [ ]:
import os

print("Current directory:", os.getcwd())
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

# Find the dataset anywhere inside MyDrive
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if "Features and Labels" in file:
            print(os.path.join(root, file))

Current directory: /content
Drive mounted: True
/content/drive/MyDrive/PJ Data (Features and Labels).csv


In [ ]:
#!/usr/bin/env python3
"""
run_analysis.py  --  ASSISTments P(J) analysis (pandas, numpy, matplotlib, scikit-learn only).

Usage:  python run_analysis.py                      (uses DATA_PATH below)
        python run_analysis.py "path/to/file.csv.gz"  (optional override)
Reads the Features-and-Labels CSV (plain or .gz; read-only) and writes ./outputs/
    tables/   CSV result tables          figures/  PNG figures
    summary.txt (text summary of ACTUAL computed results), run_log.txt, errors.log (only if a section failed)
"""
import os, sys, gzip, time, traceback, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# =============================== EDIT HERE ===================================
DATA_PATH = "/content/drive/MyDrive/PJ Data (Features and Labels).csv"
OUT_DIR = "/content/drive/MyDrive/outputs"

# Only override the path when an explicit dataset argument is provided.
if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
    DATA_PATH = sys.argv[1]
    SEED = 20261029
TEST_SIZE = 0.20
N_FOLDS = 5
MIN_STUDENT_ROWS = 20        # minimum rows per student for behavioural profiles
MAX_OPP_PLOT = 30
N_BOOT = 200

TAB, FIG = os.path.join(OUT_DIR, "tables"), os.path.join(OUT_DIR, "figures")
os.makedirs(TAB, exist_ok=True); os.makedirs(FIG, exist_ok=True)

# Columns that are NEVER used as predictors: target, BKT state/params, identifiers, timestamps, prior-model outputs.
EXCLUDE_EXACT = {"P(J)", "PJ", "Ln", "Ln-1", "G", "S", "T", "groupId", "userId", "skill", "problemId",
                 "assignmentId", "assistmentId", "startTime", "endTime", "row_id", "opp", "seqlen", "seq_id", "labeled"}
EXCLUDE_PATTERNS = ("model", "pred", "training")      # guards against prior-model prediction columns
TEXT_COLS = ["skill", "problemType"]
REQUIRED = ["userId", "skill", "startTime", "correct", "hint", "P(J)"]

LOG, LIMITS, R = [], [], {}


def log(msg):
    print(msg, flush=True); LOG.append(msg)


def save_table(df, name, index=False):
    df.to_csv(os.path.join(TAB, name), index=index)
    log(f"  table: {name} ({len(df)} rows)")


def save_fig(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIG, name), dpi=130); plt.close(fig)
    log(f"  figure: {name}")


def section(title):
    def deco(fn):
        def wrap(*a, **k):
            log(f"\n=== {title} ===")
            try:
                return fn(*a, **k)
            except Exception as e:
                tb = traceback.format_exc()
                with open(os.path.join(OUT_DIR, "errors.log"), "a") as f:
                    f.write(f"[{title}]\n{tb}\n")
                LIMITS.append(f"Section '{title}' FAILED ({type(e).__name__}: {e}); its outputs are missing/incomplete. See errors.log.")
                log(f"  !! FAILED: {e}")
                return None
        return wrap
    return deco


def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))


# ------------------------------- 1. LOAD + CLEAN -----------------------------
@section("1. Load, inventory, cleaning, missing values, label counts")
def load_and_clean():
    if not os.path.exists(DATA_PATH):
        sys.exit(f"ERROR: data file not found: {DATA_PATH}\nEdit DATA_PATH at the top of run_analysis.py.")
    opener = gzip.open if DATA_PATH.lower().endswith(".gz") else open
    with opener(DATA_PATH, "rt", errors="replace") as f:
        n_lines = sum(1 for _ in f) - 1
    raw = pd.read_csv(DATA_PATH, low_memory=False, on_bad_lines="skip", encoding_errors="replace")
    raw.columns = [str(c).strip() for c in raw.columns]
    clog = [("raw rows read", len(raw), f"file has ~{n_lines} data lines; difference = malformed lines skipped by the parser (approx.; "
                                         "embedded newlines can distort the line count)")]
    miss_req = [c for c in REQUIRED if c not in raw.columns]
    if miss_req:
        sys.exit(f"ERROR: required columns missing from file: {miss_req}\nFound: {list(raw.columns)}")

    # raw missing-value summary (before coercion; '.' in P(J) and empty strings counted as missing)
    rawm = raw.copy()
    for c in rawm.columns:
        if rawm[c].dtype == object or str(rawm[c].dtype).startswith("str"):
            rawm[c] = rawm[c].astype("string").str.strip().replace({".": pd.NA, "": pd.NA})
    ms = pd.DataFrame({"column": raw.columns, "dtype_raw": [str(raw[c].dtype) for c in raw.columns],
                       "n_missing": [int(rawm[c].isna().sum()) for c in raw.columns]})
    ms["pct_missing"] = 100 * ms.n_missing / max(len(raw), 1)
    save_table(ms.sort_values("n_missing", ascending=False), "T03_missing_values_raw.csv")

    d = raw.copy()
    # text columns: strip stray quotes/whitespace
    for c in TEXT_COLS:
        if c in d:
            d[c] = d[c].astype("string").str.replace(r"[\"']", "", regex=True).str.strip().str.lower()
    # numeric coercion (log newly-created NaN = malformed values)
    coerced = {}
    for c in d.columns:
        if c in TEXT_COLS:
            continue
        before = d[c].isna().sum()
        if c == "P(J)":
            s = d[c].astype("string").str.strip()
            R["n_dot_label"] = int((s == ".").sum())
            d["PJ"] = pd.to_numeric(s, errors="coerce")
            coerced[c] = int(d["PJ"].isna().sum() - before - R["n_dot_label"])
            continue
        d[c] = pd.to_numeric(d[c], errors="coerce")
        coerced[c] = int(d[c].isna().sum() - before)
    bad = {k: v for k, v in coerced.items() if v > 0}
    clog.append(("non-numeric / malformed values coerced to NaN", sum(bad.values()), f"by column: {bad}" if bad else "none"))
    num = d.select_dtypes(include=[np.number]).columns
    n_inf = int(np.isinf(d[num]).sum().sum())
    d[num] = d[num].replace([np.inf, -np.inf], np.nan)
    clog.append(("inf values -> NaN", n_inf, ""))

    n0 = len(d); d = d.dropna(subset=["userId", "skill", "startTime"])
    clog.append(("dropped rows missing userId/skill/startTime", n0 - len(d), "cannot be placed in a student-skill sequence"))
    n0 = len(d); d = d.drop_duplicates()
    clog.append(("dropped exact duplicate rows", n0 - len(d), ""))
    if "timeTaken" in d:
        neg = int((d.timeTaken < 0).sum()); zero = int((d.timeTaken == 0).sum())
        d.loc[d.timeTaken < 0, "timeTaken"] = np.nan
        clog.append(("negative timeTaken set to NaN", neg, f"{zero} rows have timeTaken == 0 (kept, flagged)"))
    for c in ("correct", "hint"):
        inv = int((~d[c].isin([0, 1]) & d[c].notna()).sum())
        d.loc[~d[c].isin([0, 1]), c] = np.nan
        clog.append((f"invalid {c} (not 0/1) set to NaN", inv, f"{int(d[c].isna().sum())} rows have missing {c}; excluded from regression"))
    oor = int(((d.PJ < 0) | (d.PJ > 1)).sum())
    d.loc[(d.PJ < 0) | (d.PJ > 1), "PJ"] = np.nan
    clog.append(("P(J) outside [0,1] set to NaN (treated as unlabeled)", oor, ""))
    d["labeled"] = d.PJ.notna()
    clog.append(("final cleaned rows", len(d), f"students={d.userId.nunique()}, skills={d.skill.nunique()}"))
    save_table(pd.DataFrame(clog, columns=["step", "n_rows_or_values", "note"]), "T02_cleaning_log.csv")

    # practice-opportunity index (all rows, chronological within student x skill)
    d = d.sort_values(["userId", "skill", "startTime"], kind="stable").reset_index(drop=True)
    g = d.groupby(["userId", "skill"], sort=False)
    d["opp"] = g.cumcount() + 1
    d["seqlen"] = g["opp"].transform("max")
    d["row_id"] = np.arange(len(d))

    # inventory
    rows = []
    for c in d.columns:
        s = d[c]
        role = ("target" if c in ("PJ",) else "BKT/ID/excluded" if c in EXCLUDE_EXACT else "text" if c in TEXT_COLS else "candidate predictor")
        r = dict(column=c, role=role, dtype=str(s.dtype), non_null=int(s.notna().sum()), n_missing=int(s.isna().sum()), n_unique=int(s.nunique()))
        if pd.api.types.is_numeric_dtype(s):
            r.update(min=s.min(), median=s.median(), mean=s.mean(), max=s.max())
        rows.append(r)
    save_table(pd.DataFrame(rows), "T01_dataset_inventory.csv")

    # labeled / unlabeled
    lab = pd.DataFrame({"group": ["labeled (P(J) observed)", "unlabeled (P(J) '.'/missing/out-of-range)", "total"],
                        "rows": [int(d.labeled.sum()), int((~d.labeled).sum()), len(d)],
                        "students": [d.loc[d.labeled, "userId"].nunique(), d.loc[~d.labeled, "userId"].nunique(), d.userId.nunique()]})
    lab["pct_rows"] = 100 * lab.rows / len(d)
    save_table(lab, "T04_labeled_unlabeled_counts.csv")
    ls = d.groupby("skill").agg(rows=("labeled", "size"), labeled=("labeled", "sum")).assign(share_labeled=lambda x: x.labeled / x.rows)
    save_table(ls.reset_index(), "T04b_labeled_share_by_skill.csv")
    # label selection check: labeled vs unlabeled
    cmp = []
    for c in [c for c in ["correct", "hint", "timeTaken", "opp", "seqlen", "original", "scaffold"] if c in d]:
        x, y = d.loc[d.labeled, c], d.loc[~d.labeled, c]
        sd = np.sqrt((x.var() + y.var()) / 2)
        cmp.append(dict(variable=c, labeled_mean=x.mean(), unlabeled_mean=y.mean(), std_mean_diff=(x.mean() - y.mean()) / sd if sd > 0 else np.nan))
    save_table(pd.DataFrame(cmp), "T04c_labeled_vs_unlabeled_comparison.csv")
    R.update(n_raw=int(len(raw)), n_clean=len(d), n_students=int(d.userId.nunique()), n_skills=int(d.skill.nunique()),
             n_labeled=int(d.labeled.sum()), n_unlabeled=int((~d.labeled).sum()), clog=clog, label_cmp=pd.DataFrame(cmp))
    log(f"  raw={len(raw)} clean={len(d)} labeled={d.labeled.sum()} unlabeled={(~d.labeled).sum()}")
    return d


# ------------------------------- 2. DESCRIPTIVES -----------------------------
@section("2. Descriptive tables and plots")
def descriptives(d):
    seq = d.groupby(["userId", "skill"]).agg(n_opp=("opp", "max"), correct=("correct", "mean"), hint=("hint", "mean")).reset_index()
    save_table(seq["n_opp"].describe(percentiles=[.1, .25, .5, .75, .9, .99]).to_frame("student_skill_sequence_length").reset_index(),
               "T10_practice_opportunities_summary.csv")
    spp = d.groupby("userId").size()
    save_table(spp.describe(percentiles=[.1, .25, .5, .75, .9, .99]).to_frame("rows_per_student").reset_index(), "T10b_rows_per_student_summary.csv")
    R["seq_median"] = float(seq.n_opp.median()); R["seq_mean"] = float(seq.n_opp.mean()); R["seq_max"] = int(seq.n_opp.max())

    cap = d[d.opp <= MAX_OPP_PLOT]
    byopp = cap.groupby("opp").agg(n_rows=("opp", "size"), n_labeled=("labeled", "sum"), mean_correct=("correct", "mean"),
                                   mean_hint=("hint", "mean"), median_timeTaken=("timeTaken", "median"), mean_PJ=("PJ", "mean"),
                                   median_PJ=("PJ", "median")).reset_index()
    save_table(byopp, "T11_by_practice_opportunity.csv")
    R["byopp"] = byopp

    corr_hint = d.groupby(["correct", "hint"]).agg(n=("PJ", "size"), n_labeled=("PJ", "count"), mean_PJ=("PJ", "mean"),
                                                   median_PJ=("PJ", "median"), mean_time=("timeTaken", "mean")).reset_index()
    save_table(corr_hint, "T12_correct_by_hint_cells.csv"); R["cells"] = corr_hint

    sk = d.groupby("skill").agg(rows=("opp", "size"), students=("userId", "nunique"), mean_correct=("correct", "mean"),
                                mean_hint=("hint", "mean"), median_time=("timeTaken", "median"),
                                n_labeled=("PJ", "count"), mean_PJ=("PJ", "mean")).reset_index().sort_values("rows", ascending=False)
    save_table(sk, "T13_by_skill.csv")

    cols = [c for c in ["correct", "hint", "timeTaken", "PJ", "opp", "seqlen"] if c in d]
    desc = d[cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
    desc["skew"] = d[cols].skew()
    save_table(desc.reset_index().rename(columns={"index": "variable"}), "T14_numeric_descriptives.csv")
    R["desc"] = desc

    pj = d.PJ.dropna()
    R["pj_stats"] = dict(n=len(pj), mean=pj.mean(), median=pj.median(), sd=pj.std(), p95=pj.quantile(.95), share_lt_001=(pj < .01).mean(), skew=pj.skew())
    R["rate"] = dict(correct=d.correct.mean(), hint=d.hint.mean(), time_med=d.timeTaken.median() if "timeTaken" in d else np.nan)

    # figures
    fig, ax = plt.subplots(figsize=(7, 4)); ax.hist(seq.n_opp.clip(upper=60), bins=60, color="#4C78A8")
    ax.set(xlabel="Practice opportunities per student x skill (clipped at 60)", ylabel="Sequences", title="Practice opportunities"); save_fig(fig, "F01_practice_opportunities.png")

    fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(byopp.opp, byopp.mean_correct, "o-", label="P(correct)"); ax.plot(byopp.opp, byopp.mean_hint, "s-", label="P(hint)")
    ax.set(xlabel="Practice opportunity #", ylabel="Rate", title="Correctness and hint use by opportunity"); ax.legend(); save_fig(fig, "F02_correct_hint_by_opportunity.png")

    fig, ax = plt.subplots(figsize=(7, 4)); t = d.timeTaken.dropna(); t = t[t > 0]
    ax.hist(np.log10(t), bins=80, color="#59A14F"); ax.set(xlabel="log10(timeTaken, s)", ylabel="Rows", title="Response time (first response, positive values)"); save_fig(fig, "F03_response_time.png")

    fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(byopp.opp, byopp.median_timeTaken, "o-", color="#59A14F")
    ax.set(xlabel="Practice opportunity #", ylabel="Median timeTaken (s)", title="Median response time by opportunity"); save_fig(fig, "F04_time_by_opportunity.png")

    fig, axs = plt.subplots(1, 2, figsize=(11, 4)); axs[0].hist(pj, bins=80, color="#E15759"); axs[0].set(xlabel="P(J)", ylabel="Rows", title="P(J) (labeled rows)")
    axs[1].hist(np.log10(pj.clip(lower=1e-6)), bins=80, color="#E15759"); axs[1].set(xlabel="log10 P(J) (clipped at 1e-6)", title="P(J), log scale"); save_fig(fig, "F05_PJ_distribution.png")

    fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(byopp.opp, byopp.mean_PJ, "o-", color="#E15759")
    ax.set(xlabel="Practice opportunity #", ylabel="Mean P(J)", title="Mean P(J) by opportunity (labeled rows)"); save_fig(fig, "F06_PJ_by_opportunity.png")

    fig, ax = plt.subplots(figsize=(6, 4)); lbl = [f"correct={int(a)}\nhint={int(b)}" for a, b in zip(corr_hint.correct, corr_hint.hint)]
    ax.bar(lbl, corr_hint.mean_PJ, color="#B07AA1"); ax.set(ylabel="Mean P(J)", title="Mean P(J) by first-response outcome"); save_fig(fig, "F07_PJ_by_correct_hint.png")

    fig, ax = plt.subplots(figsize=(6, 4)); ax.boxplot([d.loc[(d.PJ.notna()) & (d.hint == h), "PJ"] for h in (0, 1)], showfliers=False); ax.set_xticklabels(["hint=0", "hint=1"])
    ax.set(ylabel="P(J)", title="P(J) by hint use (outliers hidden)"); save_fig(fig, "F08_PJ_boxplot_by_hint.png")


# ------------------------------- 3. PROFILES ---------------------------------
@section("3. Student behavioural profiles (exploratory)")
def profiles(d):
    g = d.groupby("userId")
    st = pd.DataFrame({"n_rows": g.size(), "n_skills": g.skill.nunique(), "correct_rate": g.correct.mean(), "hint_rate": g.hint.mean()})
    if "frIsHelpRequest" in d: st["help_request_rate"] = g.frIsHelpRequest.mean()
    if "bottomHint" in d: st["bottom_out_rate"] = g.bottomHint.mean()
    if "timeTaken" in d:
        st["median_log_time"] = g.timeTaken.apply(lambda s: np.log1p(s.clip(lower=0)).median())
        q10 = d.timeTaken.quantile(.10)
        st["fast_response_share"] = g.timeTaken.apply(lambda s: (s <= q10).mean())
    st["mean_seq_len"] = d.groupby(["userId", "skill"]).opp.max().groupby("userId").mean()
    st["mean_PJ_labeled"] = g.PJ.mean()                       # NOT used for clustering (external check only)
    st["n_labeled"] = g.PJ.count()
    st = st[st.n_rows >= MIN_STUDENT_ROWS]
    feats = [c for c in ["correct_rate", "hint_rate", "help_request_rate", "bottom_out_rate", "median_log_time", "fast_response_share", "mean_seq_len"] if c in st]
    X = st[feats].copy(); X = X.fillna(X.median())
    Z = StandardScaler().fit_transform(X)
    if len(st) < 50:
        LIMITS.append(f"Profiles: only {len(st)} students with >= {MIN_STUDENT_ROWS} rows; profiles skipped."); return
    rng = np.random.RandomState(SEED); sil = []
    for k in range(2, 7):
        lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)
        sil.append((k, silhouette_score(Z, lab, sample_size=min(5000, len(Z)), random_state=SEED)))
    save_table(pd.DataFrame(sil, columns=["k", "silhouette"]), "T20_profile_k_selection.csv")
    k = max(sil, key=lambda t: t[1])[0]
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z); st["profile"] = km.labels_
    zc = pd.DataFrame(km.cluster_centers_, columns=feats)
    names = {}
    for i, row in zc.iterrows():
        top = row.abs().sort_values(ascending=False).index[:2]
        names[i] = f"P{i}: " + ", ".join(f"{'high' if row[t] > 0 else 'low'} {t}" for t in top)
    st["profile_name"] = st.profile.map(names)
    prof = st.groupby("profile_name").agg(n_students=("n_rows", "size"), **{f"mean_{c}": (c, "mean") for c in feats},
                                          mean_PJ_labeled=("mean_PJ_labeled", "mean")).reset_index()
    save_table(prof, "T21_profile_summary.csv"); save_table(zc.round(3).assign(profile=[names[i] for i in zc.index]), "T22_profile_centroids_z.csv")
    save_table(st.reset_index(), "T23_student_level_features_and_profile.csv")
    R["prof"] = prof; R["prof_k"] = k; R["prof_sil"] = dict(sil)[k]; R["prof_n"] = len(st)

    P = PCA(2, random_state=SEED).fit_transform(Z)
    fig, ax = plt.subplots(figsize=(7, 5)); sc = ax.scatter(P[:, 0], P[:, 1], c=km.labels_, s=6, cmap="tab10", alpha=.6)
    ax.set(xlabel="PC1", ylabel="PC2", title=f"Student profiles (k={k}, silhouette={R['prof_sil']:.2f})")
    ax.legend(*sc.legend_elements(), title="profile", loc="best"); save_fig(fig, "F20_profiles_pca.png")
    fig, ax = plt.subplots(figsize=(9, 4.5)); w = .8 / k
    for i in range(k): ax.bar(np.arange(len(feats)) + i * w, zc.loc[i], w, label=f"P{i} (n={(km.labels_ == i).sum()})")
    ax.set_xticks(np.arange(len(feats)) + .4 - w / 2); ax.set_xticklabels(feats, rotation=30, ha="right"); ax.axhline(0, c="k", lw=.5)
    ax.set(ylabel="Centroid (z-score)", title="Profile centroids"); ax.legend(); save_fig(fig, "F21_profile_centroids.png")
    fig, ax = plt.subplots(figsize=(7, 4)); ax.boxplot([st.loc[st.profile == i, "mean_PJ_labeled"].dropna() for i in range(k)], showfliers=False); ax.set_xticklabels([f"P{i}" for i in range(k)])
    ax.set(ylabel="Student mean P(J)", title="Mean P(J) by profile (external check, not used to cluster)"); save_fig(fig, "F22_PJ_by_profile.png")


# ------------------------------- 4. REGRESSION -------------------------------
def flag_leak(c):
    """Return (risk, reason). Heuristic from column names only - provenance of the P(J) label is undocumented."""
    if c.startswith("afr") or "After" in c or c.startswith("endsWith") or c in ("hintCount", "attemptCount", "bottomHint") or "Scaffolding" in c and ("afr" in c.lower()):
        return "HIGH", "summarises actions AFTER the first response / whole-problem totals; may encode the later correctness that P(J) is built from"
    if c.startswith(("fastest", "slowest")):
        return "HIGH", "min/max times over actions in the problem; may include post-first-response actions"
    if c == "hintTotal":
        return "MEDIUM", "unclear whether it totals hints over the whole problem (post-first-response)"
    if c == "correct":
        return "MEDIUM", "Ln is a deterministic update of Ln-1 using 'correct'; if P(J) is derived from Ln, correct enters the target indirectly"
    if c.startswith(("totalFr", "totalAfr", "frPast", "past8", "stl", "frTotal", "frTime")):
        return "MEDIUM", "cumulative per-skill history; proxy for BKT state (Ln-1) and may not be strictly lagged"
    return "LOW", "no mechanism identified from names alone (still unverified)"


def make_pipe(cols, kind, time_cols):
    if kind == "mean":
        return Pipeline([("i", SimpleImputer(strategy="median")), ("m", DummyRegressor(strategy="mean"))])
    if kind == "hgb":
        return Pipeline([("m", HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_leaf_nodes=31, random_state=SEED))])
    tc = [c for c in cols if c in time_cols]; oc = [c for c in cols if c not in time_cols]
    tr = []
    if tc: tr.append(("t", Pipeline([("i", SimpleImputer(strategy="median")), ("l", FunctionTransformer(slog)), ("s", StandardScaler())]), tc))
    if oc: tr.append(("o", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), oc))
    return Pipeline([("c", ColumnTransformer(tr)), ("m", Ridge(alpha=1.0))])


def metrics(y, p):
    return dict(MAE=mean_absolute_error(y, p), RMSE=float(np.sqrt(mean_squared_error(y, p))), R2=r2_score(y, p))


@section("4. Reduced vs full-feature regression (student-level split)")
def regression(d):
    a = d[d.labeled & d.correct.notna() & d.hint.notna()].copy()
    log(f"  analytic rows={len(a)}, students={a.userId.nunique()}")
    if len(a) < 200 or a.userId.nunique() < 20:
        LIMITS.append("Regression skipped: too few labeled rows/students."); return
    if "problemType" in a:
        vc = a.problemType.value_counts(normalize=True)
        for lv in vc[vc >= .01].index: a[f"pt_{lv}"] = (a.problemType == lv).astype(float)
    cand = [c for c in a.columns if pd.api.types.is_numeric_dtype(a[c]) and c not in EXCLUDE_EXACT
            and not any(p in c.lower() for p in EXCLUDE_PATTERNS)]
    cand = [c for c in cand if a[c].notna().any() and a[c].nunique() > 1]
    full = cand
    reduced = ["correct", "hint"]
    leak = pd.DataFrame([(c, *flag_leak(c)) for c in full], columns=["feature", "risk", "reason"])
    post = leak[leak.risk == "HIGH"].feature.tolist()
    nopost = [c for c in full if c not in post]
    sets = {"reduced (correct, hint)": reduced, "full (all allowed features)": full, "full minus HIGH-risk post-first-response features": nopost}
    bad = [c for s in sets.values() for c in s if c in EXCLUDE_EXACT or any(p in c.lower() for p in EXCLUDE_PATTERNS)]
    assert not bad, f"excluded column leaked into predictors: {bad}"
    time_cols = {c for c in full if any(k in c.lower() for k in ("time", "fastest", "slowest"))}
    save_table(pd.DataFrame({"set": [k for k, v in sets.items() for _ in v], "feature": [c for v in sets.values() for c in v]}), "T30_feature_sets.csv")
    log(f"  features: reduced={len(reduced)} full={len(full)} no-post={len(nopost)}")

    y = a.PJ.values; groups = a.userId.values
    tr, te = next(GroupShuffleSplit(1, test_size=TEST_SIZE, random_state=SEED).split(a, y, groups))
    assert not set(groups[tr]) & set(groups[te]), "student overlap between train and test!"
    R["split"] = dict(train_rows=len(tr), test_rows=len(te), train_students=len(set(groups[tr])), test_students=len(set(groups[te])))

    models = {"mean baseline": "mean", "Ridge (linear)": "ridge", "HistGradientBoosting": "hgb"}
    res, preds, fitted = [], pd.DataFrame({"row_id": a.row_id.values[te], "userId": groups[te], "y_true": y[te]}), {}
    for sname, cols in sets.items():
        for mname, kind in models.items():
            if kind == "mean" and sname != "reduced (correct, hint)":
                continue
            pipe = make_pipe(cols, kind, time_cols).fit(a.iloc[tr][cols], y[tr])
            p = pipe.predict(a.iloc[te][cols]); fitted[(sname, mname)] = (pipe, cols)
            preds[f"{sname}|{mname}"] = p
            res.append(dict(feature_set=sname, model=mname, n_features=len(cols), **{f"test_{k}": v for k, v in metrics(y[te], p).items()},
                            **{f"train_{k}": v for k, v in metrics(y[tr], pipe.predict(a.iloc[tr][cols])).items()}))
            log(f"  {sname} | {mname}: R2={res[-1]['test_R2']:.4f} MAE={res[-1]['test_MAE']:.5f}")
    res = pd.DataFrame(res); save_table(res, "T31_regression_test_metrics.csv")
    save_table(preds, "T32_test_predictions.csv")
    R["res"] = res

    # grouped CV (robustness)
    cv = []
    for sname, cols in sets.items():
        for mname, kind in models.items():
            if kind == "mean": continue
            for f, (i, j) in enumerate(GroupKFold(N_FOLDS).split(a, y, groups)):
                p = make_pipe(cols, kind, time_cols).fit(a.iloc[i][cols], y[i]).predict(a.iloc[j][cols])
                cv.append(dict(feature_set=sname, model=mname, fold=f, **metrics(y[j], p)))
    cv = pd.DataFrame(cv); save_table(cv, "T33_groupkfold_cv_folds.csv")
    cvs = cv.groupby(["feature_set", "model"])[["MAE", "RMSE", "R2"]].agg(["mean", "std"]); cvs.columns = ["_".join(c) for c in cvs.columns]
    save_table(cvs.reset_index(), "T34_groupkfold_cv_summary.csv"); R["cv"] = cvs.reset_index()

    # student-level bootstrap of (full - reduced) differences on the test set
    idx = {u: np.where(groups[te] == u)[0] for u in np.unique(groups[te])}; us = np.array(list(idx)); rng = np.random.RandomState(SEED); boot = []
    yt = y[te]
    for mname in ("Ridge (linear)", "HistGradientBoosting"):
        pr, pf = preds[f"reduced (correct, hint)|{mname}"].values, preds[f"full (all allowed features)|{mname}"].values
        dd = []
        for _ in range(N_BOOT):
            ii = np.concatenate([idx[u] for u in rng.choice(us, len(us))])
            dd.append((r2_score(yt[ii], pf[ii]) - r2_score(yt[ii], pr[ii]),
                       mean_absolute_error(yt[ii], pr[ii]) - mean_absolute_error(yt[ii], pf[ii])))
        dd = np.array(dd)
        boot.append(dict(model=mname, delta_R2_full_minus_reduced=r2_score(yt, pf) - r2_score(yt, pr), R2_ci95_lo=np.percentile(dd[:, 0], 2.5), R2_ci95_hi=np.percentile(dd[:, 0], 97.5),
                         MAE_reduction_full_vs_reduced=mean_absolute_error(yt, pr) - mean_absolute_error(yt, pf), MAE_ci95_lo=np.percentile(dd[:, 1], 2.5), MAE_ci95_hi=np.percentile(dd[:, 1], 97.5)))
    boot = pd.DataFrame(boot); save_table(boot, "T35_bootstrap_full_vs_reduced_student_level.csv"); R["boot"] = boot

    # leakage screen + feature importance
    Xl = a[full]; scr = []
    for c in full:
        r = dict(feature=c)
        for tgt in ("PJ", "Ln", "Ln-1"):
            r[f"abs_corr_{tgt}"] = abs(Xl[c].corr(a[tgt])) if tgt in a else np.nan
        scr.append(r)
    scr = pd.DataFrame(scr).merge(leak, on="feature")
    pipe, cols = fitted[("full (all allowed features)", "HistGradientBoosting")]
    sub = np.random.RandomState(SEED).choice(len(te), min(15000, len(te)), replace=False)
    pi = permutation_importance(pipe, a.iloc[te][cols].iloc[sub], y[te][sub], n_repeats=3, random_state=SEED, scoring="r2")
    scr = scr.merge(pd.DataFrame({"feature": cols, "perm_importance_R2_drop": pi.importances_mean}), on="feature", how="left")
    scr["empirical_flag"] = np.where((scr.abs_corr_PJ > .5) | (scr.abs_corr_Ln > .5) | (scr["abs_corr_Ln-1"] > .5), "|r|>0.5 with PJ/Ln/Ln-1", "")
    scr = scr.sort_values("perm_importance_R2_drop", ascending=False)
    save_table(scr, "T36_leakage_screen_and_importance.csv"); R["scr"] = scr

    # figures
    fig, ax = plt.subplots(figsize=(10, 4.5)); lbl = [f"{r.feature_set.split(' (')[0][:22]}\n{r.model[:12]}" for r in res.itertuples()]
    ax.bar(range(len(res)), res.test_R2, color="#4C78A8"); ax.set_xticks(range(len(res))); ax.set_xticklabels(lbl, fontsize=7, rotation=20, ha="right")
    ax.set(ylabel="Test R²", title="Student-level held-out R²"); ax.axhline(0, c="k", lw=.5); save_fig(fig, "F30_test_R2.png")
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, m in zip(axs, ("MAE", "RMSE")):
        ax.bar(range(len(res)), res[f"test_{m}"], color="#F28E2B"); ax.set_xticks(range(len(res))); ax.set_xticklabels(lbl, fontsize=6, rotation=30, ha="right"); ax.set_title(f"Test {m}")
    save_fig(fig, "F31_test_MAE_RMSE.png")
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, s in zip(axs, ("reduced (correct, hint)", "full (all allowed features)")):
        p = preds[f"{s}|HistGradientBoosting"]; ax.hexbin(yt, p, gridsize=50, bins="log", mincnt=1); lim = max(yt.max(), p.max())
        ax.plot([0, lim], [0, lim], "r--", lw=.8); ax.set(xlabel="Observed P(J)", ylabel="Predicted", title=s.split(" (")[0] + " / HGB")
    save_fig(fig, "F32_pred_vs_observed.png")
    top = scr.head(15).iloc[::-1]; col = top.risk.map({"HIGH": "#E15759", "MEDIUM": "#F28E2B", "LOW": "#4C78A8"})
    fig, ax = plt.subplots(figsize=(8, 5.5)); ax.barh(top.feature, top.perm_importance_R2_drop, color=col)
    ax.set(xlabel="Permutation importance (drop in test R²)", title="Top features, full HGB (red=HIGH, orange=MEDIUM leakage risk)"); save_fig(fig, "F33_feature_importance.png")


# ------------------------------- 5. SUMMARY ----------------------------------
@section("5. Text summary")
def write_summary():
    L = []; add = L.append
    add("P(J) ANALYSIS - SUMMARY OF ACTUAL RESULTS (auto-generated; every number below was computed by this run)")
    add(f"Data file: {DATA_PATH}\n")
    add("1. DATA")
    add(f"- Raw rows read: {R.get('n_raw')}; cleaned rows: {R.get('n_clean')}; students: {R.get('n_students')}; skills: {R.get('n_skills')}")
    add(f"- Labeled (valid P(J)): {R.get('n_labeled')}; unlabeled: {R.get('n_unlabeled')} (of which '.' placeholders in raw file: {R.get('n_dot_label')})")
    for s, n, note in R.get("clog", []): add(f"  * {s}: {n}  {note}")
    if "label_cmp" in R:
        big = R["label_cmp"][R["label_cmp"].std_mean_diff.abs() > .2]
        add("- Labeled vs unlabeled rows differ (|std diff| > 0.2) on: " + (", ".join(f"{r.variable} ({r.std_mean_diff:+.2f})" for r in big.itertuples()) or "none of the checked variables")
            + ". Missing labels are NOT shown to be random; results apply to labeled rows only.")
    add("\n2. DESCRIPTIVES")
    if "seq_median" in R: add(f"- Practice opportunities per student x skill: median {R['seq_median']:.0f}, mean {R['seq_mean']:.1f}, max {R['seq_max']}")
    if "rate" in R: add(f"- Overall first-response correct rate {R['rate']['correct']:.3f}; hint rate {R['rate']['hint']:.3f}; median timeTaken {R['rate']['time_med']:.1f}")
    if "pj_stats" in R:
        s = R["pj_stats"]; add(f"- P(J) (n={s['n']}): mean {s['mean']:.4f}, median {s['median']:.4f}, sd {s['sd']:.4f}, 95th pct {s['p95']:.4f}, share < 0.01 = {s['share_lt_001']:.3f}, skew {s['skew']:.2f}")
    if "cells" in R:
        for r in R["cells"].itertuples(): add(f"  * correct={int(r.correct)}, hint={int(r.hint)}: n={r.n}, mean P(J)={r.mean_PJ:.4f}")
    add("\n3. STUDENT PROFILES (exploratory; k-means on student-level behavioural aggregates, P(J) not used in clustering)")
    if "prof" in R:
        add(f"- {R['prof_n']} students with >= {MIN_STUDENT_ROWS} rows; k={R['prof_k']} chosen by silhouette ({R['prof_sil']:.3f}).")
        if R["prof_sil"] < .25: add("- WEAK cluster structure (silhouette < 0.25): treat profiles as a descriptive segmentation, not as distinct student types.")
        for r in R["prof"].itertuples(): add(f"  * {r.profile_name}: n={r.n_students}, mean P(J) of members={r.mean_PJ_labeled:.4f}")
    else: add("- Not produced (see limitations).")
    add("\n4. REGRESSION (student-level split; metrics on held-out students)")
    if "res" in R:
        sp = R["split"]; add(f"- Train {sp['train_rows']} rows / {sp['train_students']} students; test {sp['test_rows']} rows / {sp['test_students']} students (no student overlap, asserted).")
        for r in R["res"].itertuples(): add(f"  * {r.feature_set} | {r.model}: test MAE={r.test_MAE:.5f}, RMSE={r.test_RMSE:.5f}, R2={r.test_R2:.4f} (train R2={r.train_R2:.4f})")
        for r in R["boot"].itertuples():
            add(f"- {r.model}: full - reduced dR2 = {r.delta_R2_full_minus_reduced:+.4f} (95% student-bootstrap CI {r.R2_ci95_lo:+.4f}..{r.R2_ci95_hi:+.4f}); MAE reduction = {r.MAE_reduction_full_vs_reduced:+.5f} (CI {r.MAE_ci95_lo:+.5f}..{r.MAE_ci95_hi:+.5f})")
        cv = R["cv"]; add(f"- {N_FOLDS}-fold student-grouped CV (mean R2): " + "; ".join(f"{r.feature_set.split(' (')[0][:30]}/{r.model[:5]}={r.R2_mean:.3f}" for r in cv.itertuples()))
    else: add("- Not produced (see limitations).")
    add("\n5. POTENTIAL TARGET LEAKAGE (NOT RESOLVED)")
    add("- Excluded from predictors: P(J), Ln, Ln-1, G, S, T, groupId, all IDs/timestamps, any column named like model/pred/training, plus opp/seqlen (seqlen uses future rows).")
    add("- The P(J) construction is undocumented in the supplied files. Name-based flags below are heuristics; they are NOT proof of leakage or of its absence.")
    if "scr" in R:
        s = R["scr"]
        for lvl in ("HIGH", "MEDIUM"):
            f = s[s.risk == lvl].feature.tolist(); add(f"- {lvl}-risk features still in the FULL model" + (" (removed in the 'minus HIGH' variant)" if lvl == "HIGH" else "") + f": {', '.join(f) if f else 'none'}")
        e = s[s.empirical_flag != ""]
        add(f"- Features with |r| > 0.5 against P(J), Ln or Ln-1 (empirical screen): {', '.join(e.feature) if len(e) else 'none'}")
        add("- Even the 'minus HIGH' set keeps 'correct' and cumulative-history features, which are BKT-state proxies; a gain over the reduced model may partly reflect leakage or BKT information rather than behaviour. Not resolved.")
        top = s.head(5)
        if any("pportunit" in f for f in top.feature[:2]): add("- Top-ranked feature(s) are practice-opportunity counts. Labels are missing non-randomly by sequence position (see labeled-vs-unlabeled table), so this may reflect label-construction/position effects rather than behaviour. Not resolved.")
        add("- Top permutation importances (full HGB): " + ", ".join(f"{r.feature} ({r.perm_importance_R2_drop:.4f}, {r.risk})" for r in top.itertuples()))
    add("\n6. LIMITATIONS")
    add("- Observational data; no causal claims. Profiles are exploratory and not validated. 'skill' and 'problemType' IDs/levels: skill is excluded; problemType levels >=1% are one-hot encoded.")
    add("- Regression uses labeled rows only; medians are fit on training data only (inside pipelines). Missing values in predictors are imputed (Ridge) or handled natively (HGB).")
    add("- Single 80/20 student split plus grouped CV; hyper-parameters are fixed defaults (no tuning).")
    for m in LIMITS: add(f"- {m}")
    txt = "\n".join(L); open(os.path.join(OUT_DIR, "summary.txt"), "w", encoding="utf-8").write(txt); log("  summary.txt written")
    print("\n" + txt)


if __name__ == "__main__":
    t0 = time.time()
    df = load_and_clean()
    if df is not None:
        descriptives(df); profiles(df); regression(df)
    write_summary()
    open(os.path.join(OUT_DIR, "run_log.txt"), "w").write("\n".join(LOG) + f"\nDone in {time.time() - t0:.0f}s")
    print(f"\nDone in {time.time() - t0:.0f}s. Outputs in ./{OUT_DIR}/")


=== 1. Load, inventory, cleaning, missing values, label counts ===
  table: T03_missing_values_raw.csv (69 rows)
  table: T02_cleaning_log.csv (10 rows)
  table: T01_dataset_inventory.csv (74 rows)
  table: T04_labeled_unlabeled_counts.csv (3 rows)
  table: T04b_labeled_share_by_skill.csv (55 rows)
  table: T04c_labeled_vs_unlabeled_comparison.csv (7 rows)
  raw=187312 clean=187312 labeled=119141 unlabeled=68171

=== 2. Descriptive tables and plots ===
  table: T10_practice_opportunities_summary.csv (11 rows)
  table: T10b_rows_per_student_summary.csv (11 rows)
  table: T11_by_practice_opportunity.csv (30 rows)
  table: T12_correct_by_hint_cells.csv (3 rows)
  table: T13_by_skill.csv (55 rows)
  table: T14_numeric_descriptives.csv (6 rows)
  figure: F01_practice_opportunities.png
  figure: F02_correct_hint_by_opportunity.png
  figure: F03_response_time.png
  figure: F04_time_by_opportunity.png
  figure: F05_PJ_distribution.png
  figure: F06_PJ_by_opportunity.png
  figure: F07_PJ_by_co

In [10]:
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

TB = OUT_DIR + "/tables/"; rng = np.random.RandomState(1)
st = pd.read_csv(TB + "T23_student_level_features_and_profile.csv")

# ---- rebuild attempt-level data (same ordering as notebook) ----
raw = pd.read_csv(DATA_PATH, low_memory=False, on_bad_lines="skip"); raw.columns = [c.strip() for c in raw.columns]
d = raw[["userId","skill","startTime","correct","hint","timeTaken","P(J)"]].copy()
d["PJ"] = pd.to_numeric(d["P(J)"].astype(str).str.strip().replace(".", np.nan), errors="coerce")
for c in ["correct","hint","timeTaken"]: d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.sort_values(["userId","skill","startTime"], kind="stable").reset_index(drop=True)
d["opp"] = d.groupby(["userId","skill"]).cumcount() + 1
d = d.merge(st[["userId","profile"]], on="userId")
d["error"] = 1 - d.correct

# ---- RQ1/RQ2: temporal curves per profile (opp 1-10) ----
cur = (d[d.opp <= 10].groupby(["profile","opp"])
       .agg(hint=("hint","mean"), error=("error","mean"), time=("timeTaken","median")).reset_index())
print(cur.pivot(index="opp", columns="profile", values=["hint","error","time"]).round(3))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for p, g in cur.groupby("profile"):
    for a, m in zip(ax, ["hint","error","time"]): a.plot(g.opp, g[m], marker="o", label=f"P{p}"); a.set_title(m); a.set_xlabel("opportunity")
ax[0].legend(); fig.tight_layout(); fig.savefig(OUT_DIR + "/figures/F23_temporal_by_profile.png", dpi=130)

# ---- RQ2: Mann-Whitney at student level, incl. spikiness ----
sk = (d.dropna(subset=["PJ"]).groupby(["userId","skill"]).PJ.agg(["max","mean","count"]).reset_index())
sk = sk[(sk["count"] >= 2) & (sk["mean"] > 0)]; sk["spike"] = sk["max"] / sk["mean"]
st = st.merge(sk.groupby("userId").spike.mean().rename("spikiness"), on="userId", how="left")
st["error_rate"] = 1 - st.correct_rate
meas = [c for c in ["hint_rate","error_rate","help_request_rate","median_log_time","mean_seq_len","spikiness","mean_PJ_labeled"] if c in st]
rows = []
for m in meas:
    a = st.loc[st.profile == 0, m].dropna(); b = st.loc[st.profile == 1, m].dropna()
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    rows.append(dict(measure=m, n0=len(a), n1=len(b), med0=a.median(), med1=b.median(), U=U, p=p, rank_biserial=2*U/(len(a)*len(b)) - 1))
mw = pd.DataFrame(rows).sort_values("p")
mw["p_holm"] = np.minimum(1, mw.p * (len(mw) - np.arange(len(mw)))).cummax()   # Holm
print(mw.round(4).to_string(index=False)); mw.to_csv(TB + "T40_mannwhitney.csv", index=False)

# ---- RQ3/H4: reduced vs full within profile (held-out students) ----
pr = pd.read_csv(TB + "T32_test_predictions.csv").merge(st[["userId","profile"]], on="userId")
def cols(model): return ("reduced (correct, hint)|" + model, "full (all allowed features)|" + model)
def stats(df, model):
    r, f = cols(model); y = df.y_true.values
    return dict(R2_red=r2_score(y, df[r]), R2_full=r2_score(y, df[f]),
                r_red=np.corrcoef(y, df[r])[0,1], r_full=np.corrcoef(y, df[f])[0,1])
def boot_delta(g, model, B=500):
    r, f = cols(model); idx = {u: x.index.values for u, x in g.groupby("userId")}; us = np.array(list(idx)); out = []
    for _ in range(B):
        ii = np.concatenate([idx[u] for u in rng.choice(us, len(us))]); s = g.loc[ii]
        out.append(r2_score(s.y_true, s[f]) - r2_score(s.y_true, s[r]))
    return np.array(out)
res, bs = [], {}
for model in ["Ridge (linear)", "HistGradientBoosting"]:
    for p in [0, 1]:
        g = pr[pr.profile == p].reset_index(drop=True); s = stats(g, model); b = boot_delta(g, model); bs[(model, p)] = b
        res.append(dict(model=model, profile=p, n_students=g.userId.nunique(), n_rows=len(g), **s,
                        dR2=s["R2_full"] - s["R2_red"], ci_lo=np.percentile(b, 2.5), ci_hi=np.percentile(b, 97.5)))
    diff = bs[(model, 0)] - bs[(model, 1)]
    print(model, "gap(P0) - gap(P1): mean %.4f  95%% CI [%.4f, %.4f]" % (diff.mean(), *np.percentile(diff, [2.5, 97.5])))
print(pd.DataFrame(res).round(4).to_string(index=False)); pd.DataFrame(res).to_csv(TB + "T41_gap_by_profile.csv", index=False)


          hint         error           time        
profile      0      1      0      1       0       1
opp                                                
1        0.061  0.281  0.360  0.663  28.062  20.880
2        0.053  0.334  0.281  0.607  19.247  12.401
3        0.046  0.334  0.273  0.609  17.582  11.173
4        0.044  0.328  0.267  0.592  16.641   9.867
5        0.043  0.318  0.278  0.636  17.049   9.465
6        0.048  0.355  0.262  0.606  16.235   8.360
7        0.051  0.336  0.290  0.681  16.188   8.797
8        0.046  0.339  0.278  0.627  15.976   9.086
9        0.046  0.324  0.284  0.662  15.834   8.496
10       0.051  0.366  0.276  0.662  15.379   7.837
          measure   n0  n1   med0    med1        U      p  rank_biserial  p_holm
help_request_rate 1794 313 0.0333  0.3654  15892.5 0.0000        -0.9434  0.0000
        hint_rate 1794 313 0.0277  0.3167  21506.5 0.0000        -0.9234  0.0000
       error_rate 1794 313 0.2812  0.6481  22248.5 0.0000        -0.9208  0.0000
